# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [2]:
import pandas as pd

df = pd.read_csv("tic-tac-toe.csv")

df.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [3]:
print(df.shape)

df.info()

print(df.isnull().sum())

(958, 10)
<class 'pandas.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   TL      958 non-null    str  
 1   TM      958 non-null    str  
 2   TR      958 non-null    str  
 3   ML      958 non-null    str  
 4   MM      958 non-null    str  
 5   MR      958 non-null    str  
 6   BL      958 non-null    str  
 7   BM      958 non-null    str  
 8   BR      958 non-null    str  
 9   class   958 non-null    bool 
dtypes: bool(1), str(9)
memory usage: 76.8 KB
TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64


In [4]:
mapping = {'x': 1, 'o': -1, 'b': 0}

# Convert board positions to numbers
board_cols = df.columns.drop('class')
df[board_cols] = df[board_cols].apply(lambda col: col.map(mapping))

# Convert target column to integer
df['class'] = df['class'].astype(int)

df.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,1,1,1,1,-1,-1,1,-1,-1,1
1,1,1,1,1,-1,-1,-1,1,-1,1
2,1,1,1,1,-1,-1,-1,-1,1,1
3,1,1,1,1,-1,-1,-1,0,0,1
4,1,1,1,1,-1,-1,0,-1,0,1


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [5]:
X = df.drop(columns=['class'])
y = df['class']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (958, 9)
y shape: (958,)


In [6]:
# Normalize features to the range [0, 1]
X_normalized = (X + 1) / 2

print(X_normalized.head())

    TL   TM   TR   ML   MM   MR   BL   BM   BR
0  1.0  1.0  1.0  1.0  0.0  0.0  1.0  0.0  0.0
1  1.0  1.0  1.0  1.0  0.0  0.0  0.0  1.0  0.0
2  1.0  1.0  1.0  1.0  0.0  0.0  0.0  0.0  1.0
3  1.0  1.0  1.0  1.0  0.0  0.0  0.0  0.5  0.5
4  1.0  1.0  1.0  1.0  0.0  0.0  0.5  0.0  0.5


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_normalized,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (766, 9)
X_test: (192, 9)
y_train: (766,)
y_test: (192,)


In [8]:
import keras

model = keras.Sequential([
    keras.Input(shape=(9,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(2, activation="softmax"),
])

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 882 (3.45 KB)

 Trainable params: 882 (3.45 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:

history = model.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Training accuracy:", history.history["accuracy"][-1])
print("Validation accuracy:", history.history["val_accuracy"][-1])


Training accuracy: 1.0
Validation accuracy: 1.0


In [11]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test loss:", loss)
print("Test accuracy:", accuracy)


Test loss: 0.10685837268829346
Test accuracy: 0.9739583134651184


In [12]:
model.save("tic-tac-toe.keras")


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [15]:
loaded_model = keras.models.load_model("tic-tac-toe.keras")

In [16]:
# 5 random games from the test set
X_sample = X_test.sample(5, random_state=42)

predictions = loaded_model.predict(X_sample, verbose=0)

predicted_classes = predictions.argmax(axis=1)

actual_classes = y_test.loc[X_sample.index]

for predicted, actual in zip(predicted_classes, actual_classes):
    print(f"Predicted: {predicted}, Actual: {actual}")


Predicted: 1, Actual: 1
Predicted: 1, Actual: 1
Predicted: 1, Actual: 1
Predicted: 1, Actual: 1
Predicted: 1, Actual: 1


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [17]:
# Model with extra hidden layer
model2 = keras.Sequential([
    keras.Input(shape=(9,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(2, activation="softmax")
])

model2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model2.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Original validation accuracy:", history.history["val_accuracy"][-1])

print("New validation accuracy:", history2.history["val_accuracy"][-1])

print("Original validation loss:", history.history["val_loss"][-1])

print("New validation loss:", history2.history["val_loss"][-1])

Original validation accuracy: 1.0
New validation accuracy: 0.9870129823684692
Original validation loss: 0.021068790927529335
New validation loss: 0.02201257459819317


In [18]:
# Model with a higher learning rate
model3 = keras.Sequential([
    keras.Input(shape=(9,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(2, activation="softmax")
])

model3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history3 = model3.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Original validation accuracy:", history.history["val_accuracy"][-1])
print("New validation accuracy:", history3.history["val_accuracy"][-1])

print("Original validation loss:", history.history["val_loss"][-1])
print("New validation loss:", history3.history["val_loss"][-1])


Original validation accuracy: 1.0
New validation accuracy: 0.9675324559211731
Original validation loss: 0.021068790927529335
New validation loss: 0.10122345387935638


In [19]:
# Model with more epochs
model4 = keras.Sequential([
    keras.Input(shape=(9,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(2, activation="softmax")
])

model4.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history4 = model4.fit(
    X_train,
    y_train,
    epochs=250,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Original validation accuracy:", history.history["val_accuracy"][-1])
print("New validation accuracy:", history4.history["val_accuracy"][-1])

print("Original validation loss:", history.history["val_loss"][-1])
print("New validation loss:", history4.history["val_loss"][-1])


Original validation accuracy: 1.0
New validation accuracy: 0.9935064911842346
Original validation loss: 0.021068790927529335
New validation loss: 0.025617828592658043


**Which approach(es) did you find helpful to improve your model performance?**

Three approaches were tried out to improve neural network: adding another hidden layer, increasing the learning rate, and increasing the number of epochs.

None of these changes improved the validation results. The original model with two hidden layers, a learning rate of 0.001, and 150 epochs performed best, with 100% validation accuracy and a validation loss of 0.0211.

Thus, adding more layers or increasing the learning rate and epochs does not always improve a neural network. Sometimes a simpler model works better.